# Basic example with a MODFLOW simulation

Generate a Voronoi grid and use it in a MODFLOW 6 simulation with two constant-head sides, a fixed-head well at (25, 25), and an impermeable fault. The barrier cells below show source layout choices, optional repairs, two-sided tracking and assignment to an HFB package.

Install the example dependencies in the notebook kernel's environment with `python -m pip install -e ".[examples]"` from the repository root. Install the MODFLOW 6 executable separately with `get-modflow /path/to/bin --subset mf6`; add that directory to PATH or set `MF6_EXE` to the executable's full path. The notebook does not download executables.

In [ ]:
import os
import shutil
import tempfile

import numpy as np
from shapely.geometry import LineString, Point, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator
from vorflow.utils import build_connectivity, cells_on, faces_on

import flopy
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

exe_request = os.environ.get("MF6_EXE", "mf6")
mf6_exe = shutil.which(exe_request)
if mf6_exe is None:
    raise FileNotFoundError(
        f"MODFLOW 6 executable not found: {exe_request!r}. "
        "Install it separately with get-modflow /path/to/bin --subset mf6, "
        "then set MF6_EXE or add its directory to PATH."
    )

## Definition of the conceptual model

Add the domain, a fixed-head well point and a fault through the existing `add_polygon()`, `add_point()` and `add_line()` API. `is_barrier=True` guides mesh geometry; hydraulic resistance is assigned separately in FloPy.

In the fault definition below:

- `quad_buffer=False` uses straddle point pairs. Set it to `True` for a one-row Gmsh quad buffer instead, then rerun the cells below.
- `barrier_policy="preserve"`, `barrier_mirrors=False` keeps the source layout. This straight fault aligns without repairs in both layouts.
- Optional repairs: set `barrier_mirrors=True` to add reflected generators, or `barrier_policy="cut"` to allow clipping. With clipping, `barrier_merge="restore"` attempts checked merges and restores failed parent cuts. `"checked"` retains refused fragments; `"none"` disables merging.

These are explicit example settings. Existing calls without these options keep the library's historical mirror/cut and legacy-merge defaults. Bends, ends and interacting features still need the diagnostics below.

In [ ]:
# Definition of geometries
domain = box(0, 0, 200, 200)
well_point = Point(25, 25)
fault_line = LineString([(100, 0), (100, 200)])

# The constant-head sides are only used to select cells after meshing,
# so they are not added to the conceptual model.
left_line = LineString([(0, 0), (0, 200)])
right_line = LineString([(200, 0), (200, 200)])

# Creation of the ConceptualMesh and addition of the geometries
cm = ConceptualMesh()
cm.add_polygon(domain, zone_id=1)
cm.add_point(
    well_point,
    point_id="Well-A",
    resolution=2,
    growth_factor=1.2,
    label="Well (5 m)"
)
cm.add_line(
    fault_line,
    line_id="Fault-1",
    resolution=1,
    is_barrier=True,
    quad_buffer=False,          # True selects the one-row quad-buffer alternative.
    quad_buffer_thickness=1,
    barrier_policy="preserve", # Use "cut" for optional clipping.
    barrier_mirrors=False,      # True adds optional reflected generators.
    barrier_merge="restore",   # Only used when the barrier is clipped.
    simplify_tolerance=0,
    label="Fault"
)

## Create the `MeshGenerator` object
We will create a `MeshGenerator` object and add our conceptual model to it. Once that is done, we can generate the triangular mesh with the `.generate()` method.


In [ ]:
mg = MeshGenerator(background_lc=100, conceptual_mesh=cm)
mg.generate()

## Create the `VoronoiTessellator` object
We can now create the Voronoi grid by creating a `VoronoiTessellator` object and adding our mesh and conceptual model to it.

In [ ]:
vt = VoronoiTessellator(mg, cm, clip_to_boundary=True)
gdf = vt.generate()

We can now plot the Voronoi mesh and the conceptual model with the `.plot()` method.

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
vt.plot(ax=ax, color='w', column=None, edgecolor='k')
cm.plot(ax=ax, alpha=0)
ax.legend()
plt.show()

## Inspect the fault's cells and connections

Track the cells on the two sides of each matched fault segment. Left/right follows the cleaned line direction: west/east for this upward fault. Cell IDs are **row positions**, matching DISV cell numbers. Deduplicate pairs because one pair may have several segment records.

The example fault reaches both domain boundaries, so its selected shared faces are complete. An internal endpoint can cut through a shared face; an HFB would then extend beyond that endpoint.

Check full target coverage before assigning the impermeable HFB; a partial match would leave hydraulic gaps. If you change the geometry and this check fails, inspect `tracking["unmatched"]` and `tracking["crossings"]`. Optional `vt.barrier_report(approximate=True)` also proposes existing shared faces in `approximate_faces`; review their `usable` flags, offsets and route topology before choosing an approximation.

The quality report checks all cells and shared faces. `ortho_error` is the angular departure from perpendicularity (degrees); `midpoint_skew` measures the connector intersection's offset from the face midpoint divided by face length (0.5 reaches an endpoint). Warnings are separate from hard geometry failures and do not establish flow accuracy.

In [ ]:
quality = vt.quality_report(domain=domain)
assert quality["summary"]["hard_valid"], quality["summary"]
print("Quality warnings:", quality["summary"]["warning_counts"])
display(quality["faces"][["ortho_error", "midpoint_skew"]].describe().loc[["50%", "max"]])

tracking = vt.barrier_report()
display(tracking["summary"].drop(columns="geometry"))
fault_summary = tracking["summary"].loc[tracking["summary"]["line_id"] == "Fault-1"]
assert (fault_summary["unmatched_length"] < 1e-6).all(), "Inspect unmatched fault intervals."
assert (fault_summary["crossing_cells"] == 0).all(), "The fault still enters cell interiors."
fault_faces = tracking["faces"].loc[tracking["faces"]["line_id"] == "Fault-1"]
assert len(fault_faces) and fault_faces["centres_opposite"].all()
fault_pairs = fault_faces[["left_cell", "right_cell"]].drop_duplicates()
print(f"{len(fault_pairs)} horizontal pairs will receive an HFB in layer 0.")
display(fault_pairs.head())

# A lightweight lookup gives the same pairs; the report additionally checks gaps.
fault_connections = faces_on(build_connectivity(gdf), fault_line).drop_duplicates(
    subset=["cell_id_1", "cell_id_2"]
)
tracked_pairs = {tuple(sorted(pair)) for pair in fault_pairs.to_numpy(dtype=int)}
lookup_pairs = {tuple(sorted(pair)) for pair in fault_connections[["cell_id_1", "cell_id_2"]].to_numpy(dtype=int)}
assert lookup_pairs == tracked_pairs

Blue and orange show the selected left/right cells; black lines join their numerical centres. The gold line is the target fault. This close view lets you check that each selected connection crosses a matched face. The HFB lookup below is checked against these same pairs.

In [ ]:
centres = gdf[["x", "y"]].to_numpy()
pair_ids = fault_pairs.to_numpy(dtype=int)
fig, ax = plt.subplots(figsize=(7, 6))
gdf.plot(ax=ax, facecolor="0.93", edgecolor="0.6", linewidth=0.5)
gdf.iloc[np.unique(pair_ids[:, 0])].plot(ax=ax, color="tab:blue", edgecolor="0.4")
gdf.iloc[np.unique(pair_ids[:, 1])].plot(ax=ax, color="tab:orange", edgecolor="0.4")
ax.add_collection(LineCollection(centres[pair_ids], colors="black", linewidths=1))
ax.scatter(*centres[np.unique(pair_ids)].T, s=12, color="black", zorder=4)
cm.clean_lines.plot(ax=ax, color="gold", linewidth=2, zorder=3)
ax.set(xlim=(97, 103), ylim=(70, 78), xlabel="Easting (m)", ylabel="Northing (m)",
       title="Fault detail: left blue / right orange; generator connections black")
ax.set_aspect("equal")
plt.show()

## Create the `MODFLOW` model using `flopy`

We will start by setting up the MODFLOW simulation using ``flopy``.

In [ ]:
# Set up the FloPy MODFLOW 6 simulation in a temporary workspace
workspace = tempfile.TemporaryDirectory()
name = "voronoi_sim"
sim = flopy.mf6.MFSimulation(sim_name=name, exe_name=mf6_exe, sim_ws=workspace.name)
tdis = flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[(1.0, 1, 1.0)])
ims = flopy.mf6.ModflowIms(sim, complexity="MODERATE")

# Set up the Groundwater Flow (GWF) and Initial Conditions (IC) packages
gwf = flopy.mf6.ModflowGwf(sim, modelname=name)
ic = flopy.mf6.ModflowGwfic(gwf, strt=20.0) # Initial head of 20 m for all cells

# Set up basic Node Property Flow (required for flow calculations and HFB)
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,
    k=10.0,      # Default hydraulic conductivity
    save_flows=True
)

# Set up the Output Control Package (OC)
head_filename = f"{gwf.name}.hds"
budget_filename = f"{gwf.name}.cbc"

oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=[head_filename],
    budget_filerecord=[budget_filename],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")]
)

The `DISV` package can be built either using `.to_disv()`, which returns the parameters needed for `flopy.mf6.ModflowGwfdisv()`, or using `.to_modflowgwfdisv(gwf, top=..., botm=...)`. Here, the first approach is demonstrated to visualize what is going on automatically within `.to_modflowgwfdisv(gwf, top=..., botm=...)`.

In [ ]:
# Build the DISV input from the VoronoiTessellator object
grid = vt.to_disv()

# Set up the DISV Package
nlay = 1  # Number of vertical layers
top = 100.0
botm = [0.0]  # Bottom elevation for layer 1

disv = flopy.mf6.ModflowGwfdisv(
    gwf,
    nlay=nlay,
    ncpl=grid["ncpl"],
    nvert=grid["nvert"],
    top=top,
    botm=botm,
    vertices=grid["vertices"],
    cell2d=grid["cell2d"]
)

# The code above can be replaced with
# disv = vt.to_modflowgwfdisv(gwf, top=top, botm=botm, nlay=1)

print(f"Created the DISV package with {grid['ncpl']} cells and {grid['nvert']} vertices.")

## Model boundary conditions
Now, the boundary conditions can be added to the model by first locating the boundary condition cells and then passing the conditions to the CHD and HFB packages.

In [ ]:
# MODFLOW cell ids are zero-based row positions in the grid, not index labels.
# Left Boundary Cells (20 m head)
left_cells = cells_on(gdf, left_line)["cell_id"].to_numpy()

# Right Boundary Cells (10 m head)
right_cells = cells_on(gdf, right_line)["cell_id"].to_numpy()

# Well Cell (5 m head)
# Find the single cell containing the well point
well_cells = cells_on(gdf, well_point)
assert len(well_cells) == 1, "Choose explicitly if the point touches several cells."
well_cell = int(well_cells["cell_id"].iloc[0])

In [ ]:
# Assign each tracked horizontal pair once in this one-layer DISV model.
# FloPy uses zero-based (layer, cell) IDs; left/right is only for inspection.
hydraulic_characteristic = 0.0  # K_barrier / barrier_width (1/day); zero blocks flow.
hfb_data = [
    ((0, int(left)), (0, int(right)), hydraulic_characteristic)
    for left, right in fault_connections[["cell_id_1", "cell_id_2"]].itertuples(index=False, name=None)
]

In [ ]:
# Consolidate all constant head boundary data into a single list
chd_period_data = []

# Add left boundary cells (20 m)
for cid in left_cells:
    chd_period_data.append(((0, cid), 20.0))

# Add right boundary cells (10 m)
for cid in right_cells:
    chd_period_data.append(((0, cid), 10.0))

# Add well cell (5 m)
chd_period_data.append(((0, well_cell), 5.0))

# Set up the CHD Package
chd = flopy.mf6.ModflowGwfchd(
    gwf, 
    stress_period_data={0: chd_period_data},
    pname="chd_main" # Explicit name prevents random index suffixes
)

# Set up the HFB Package
hfb = flopy.mf6.ModflowGwfhfb(
    gwf,
    print_input=True,
    maxhfb=len(hfb_data),
    stress_period_data={0: hfb_data}
)

## Simulation

The model is now ready and can be run.

In [ ]:
# Write and run the simulation
sim.write_simulation()
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not run successfully:\n" + "\n".join(buff)

## Results

The heads can now be read and added to the grid GeoDataFrame for plotting.

In [ ]:
# Read the heads of the only stress period and time step
heads = gwf.output.head().get_data(kstpkper=(0, 0))
assert heads.shape[0] == nlay and heads[0].size == len(gdf)
assert np.isfinite(heads).all()
assert heads.min() >= 5.0 - 1e-5 and heads.max() <= 20.0 + 1e-5

# The heads are ordered by icell2d; to_disv()["node_id"] maps them back to grid cells
layer_0_heads = dict(zip(grid["node_id"], heads[0].ravel()))
gdf["hydraulic_head"] = gdf["node_id"].map(layer_0_heads)

# Generate the plot using GeoPandas
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the Voronoi polygons colored by their hydraulic head values
gdf.plot(
    column="hydraulic_head",
    cmap="viridis",
    edgecolor="black",
    linewidth=0.3,
    alpha=0.9,
    legend=True,
    legend_kwds={"label": "Hydraulic Head (m)"},
    ax=ax
)

# Formatting
ax.set_ylabel("Northing (m)")
ax.set_xlabel("Easting (m)")
ax.set_xlim(0, 200)
ax.set_ylim(0, 200)
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

# Remove the temporary model workspace
workspace.cleanup()